# Chapter 4 (part 1) — Simple and Multiple Linear Regression

Covers the first part of Ch 4 of *Practical Statistics for Data Scientists*: Simple Linear Regression (equation, fitted values, residuals) → Multiple Linear Regression → Assessing the Model (RMSE, R², adjusted R²) → Model Selection and Stepwise Regression → Weighted Regression, all on Ames Housing with `Sale_Price` as the target.

The thread running through this notebook: **a regression is a machine for answering "how much does y move when x moves a bit, with everything else held where it is?"** — and almost every pitfall (coefficients that change when you add a variable, R² that flatters you, stepwise results that don't replicate) comes from forgetting what "everything else" means or what the model was allowed to see.

Ask for a hint if you get stuck; per the repo's review protocol, the first couple of asks per question get a nudge, not the answer. The exam at the bottom is the part for checking whether the intuition stuck.

Primary tools: `statsmodels.formula.api.ols` for fitting and reading summaries, pandas / numpy for the by-hand checks, `sklearn.model_selection.train_test_split` for hold-out splits.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape

## 1. Simple Linear Regression

Simple linear regression draws the one straight line `Sale_Price ≈ b0 + b1 * Gr_Liv_Area` that makes the squared vertical misses as small as possible.

Numeric picture first. Take three houses: (1000 sqft, $100k), (1500 sqft, $140k), (2000 sqft, $200k). The average house is 1500 sqft and $146,667. The slope is "how do the dollar-gaps from average line up with the sqft-gaps from average?" — houses are -500, 0, +500 sqft from the mean and -46,667, -6,667, +53,333 dollars from the mean. Multiply paired gaps and add: (-500)(-46,667) + 0 + (500)(53,333) = 50,000,000; divide by the sum of squared sqft-gaps (250,000 + 0 + 250,000 = 500,000) and you get b1 = $100 per sqft. The intercept is whatever makes the line pass through the average point: 146,667 - 100(1500) = -$3,333. The *fitted value* for a house is the line's height at its sqft; the *residual* is actual minus fitted.

**Question 1a.** Fit `Sale_Price ~ Gr_Liv_Area` on the full data with `smf.ols`. Report the slope and intercept in plain dollars-per-square-foot terms. Does the intercept mean anything you'd say out loud? Then confirm, *without* using the fit, that the slope equals `corr(x, y) * std(y) / std(x)` and also `cov(x, y) / var(x)` — and say in a sentence why correlation and slope are different numbers.

*Tools for this question:*
- `smf.ols("Sale_Price ~ Gr_Liv_Area", data=df).fit()` — formula is `y ~ x1 + x2`, the intercept is added automatically. Read results with `.params` (coefficients), `.bse` (standard errors), `.summary()`.
- `s1.corr(s2)` Pearson r; `s.std()` / `s.var()` use n-1 by default; `s1.cov(s2)` covariance.

**Question 1b.** From the fitted model pull the residuals. Check numerically: (i) they sum to (essentially) zero, (ii) they're uncorrelated with `Gr_Liv_Area`. These aren't coincidences — in a sentence or two, explain why the least-squares line is *forced* to have both properties. Then: which three houses have the biggest positive residuals and which three the biggest negative? What's different about them in terms of columns you haven't put in the model (look at `Neighborhood`, `Overall_Qual`, `Sale_Condition`)?

*Tools for this question:*
- `res.resid` residuals and `res.fittedvalues` fitted values (both indexed like `df`); `np.corrcoef(a, b)[0, 1]` for the correlation.
- `df.assign(resid=res.resid).nlargest(3, "resid")` / `.nsmallest(3, "resid")` to pull the extreme rows.

## 2. Assessing the Model — RMSE and R²

Two summaries of "how good is this line":

- **RMSE** (root mean squared error) is the typical size of a miss, in the target's units. If three residuals are +30k, -40k, +50k, square them (900M, 1.6B, 2.5B), average (1.67B), take the root: about $40.8k. Squaring means a single $200k miss counts four times what a $100k miss does, so RMSE is more sensitive to big misses than a plain average absolute miss.
- **R²** is the share of the target's wiggle that the model accounts for: `1 - (sum of squared residuals) / (sum of squared gaps from the plain mean)`. Predicting everyone at the average price is the "know-nothing" benchmark, and R² measures how much better than that benchmark you did. R² = 0 means no better than the mean; R² = 1 means perfect.

(`statsmodels` reports a *residual standard error* that divides by n - p instead of n; for n = 2,930 it's nearly identical to RMSE.)

**Question 2a.** For the model from Question 1: compute RMSE and R² *from the residuals by hand* (don't read them off the summary), then confirm they match `.rsquared` and `np.sqrt(.mse_resid)`. Compare the RMSE to the standard deviation of `Sale_Price` itself: what is the model buying you, in dollars? Is a typical miss of that size good or bad? What would you need to know to decide?

*Tools for this question:*
- `res.rsquared`; `np.sqrt(res.mse_resid)` (divides by n-p, so it differs slightly from `np.sqrt(np.mean(res.resid**2))`, which divides by n).
- By hand: `sst = ((y - y.mean())**2).sum()`, `sse = (res.resid**2).sum()`, `r2 = 1 - sse/sst`.

**Question 2b.** RMSE and R² can tell different stories. Fit the same model on `log(Sale_Price)` (use `np.log(Sale_Price)` in the formula). What happened to R²? Can you compare this model's RMSE to the one from 2a? Why or why not? (Don't convert yet — just think about what units the residuals are in.) Put your reasoning in a markdown cell.

*Tools for this question:*
- Transforms work inside the formula: `smf.ols("np.log(Sale_Price) ~ Gr_Liv_Area", data=df)`.

_Your reasoning:_

## 3. Multiple Linear Regression

Add more predictors and the model fits a *plane* (or hyperplane): `Sale_Price ≈ b0 + b1*Gr_Liv_Area + b2*Year_Built + ...`. The crucial change in meaning: each coefficient is now the effect of that variable **holding the others fixed**. Numeric picture: if the model says $90/sqft for living area and $700/year for build year, then two houses that are the same age but differ by 100 sqft are predicted to differ by $9,000 — and two houses with identical sqft built 10 years apart differ by $7,000. Each slope answers its own "all else equal" question.

**Question 3.** Fit `Sale_Price ~ Gr_Liv_Area + Year_Built + Garage_Area + Total_Bsmt_SF`. (a) Compare the `Gr_Liv_Area` coefficient to the one in Question 1. Did it move? Which direction, and why would adding *Year_Built* and *Total_Bsmt_SF* change what "an extra square foot of living area" is worth? (b) Interpret the `Year_Built` coefficient in dollars in one sentence. (c) The intercept is a huge negative number. Why — and what does it say about whether the intercept is something to interpret? (d) Compute RMSE and R² for this model. By how much did RMSE improve over Question 2a — a lot, a little?

*Tools for this question:*
- Add predictors with `+`: `"Sale_Price ~ A + B + C"`. `res.summary().tables[1]` prints just the coefficient table.

_Your interpretation:_

## 4. In-sample fit vs. out-of-sample fit; adjusted R²

R² can only go up when you add a variable — even a useless one — because the least-squares fit can always use a bit of noise to shave the training residuals. Here's the numeric picture: with 3 data points a line can't generally hit all three, but with 3 data points and 2 predictors plus an intercept you can hit them *exactly* (R² = 1), no matter what the predictors are. In general, as the number of columns approaches the number of rows, the model can memorize.

**Adjusted R²** charges a toll for each added predictor: `1 - (1 - R²) * (n - 1) / (n - p - 1)`. If a predictor doesn't improve fit by more than chance would, adjusted R² falls.

**Question 4.** Split the data into 70% train / 30% test (`train_test_split` with a fixed `random_state`). On the train set, start from `Sale_Price ~ Gr_Liv_Area + Year_Built` and add pure-noise columns (`np.random.normal`, new column each time) in batches: 0, 10, 30, 100, 300 columns. For each batch record **train R², train adjusted R², and test RMSE** (predict on the test set with `.predict`). Make a small table or plot.
(a) Describe in words what each of the three quantities does as the number of noise columns grows. (b) At what point, if any, does adjusted R² fail to warn you? (c) What's the single-sentence moral about reporting R² computed on the same data used for fitting?

*Tools for this question:*
- `train_test_split(df, test_size=0.3, random_state=0)` returns `(train, test)`; `res.predict(test_df)` returns predictions for new rows; `res.rsquared_adj`.
- Noise columns: `rng = np.random.default_rng(0); rng.normal(size=len(train))`. Build the formula from a list: `"Sale_Price ~ " + " + ".join(cols)`.

_Your interpretation:_

## 5. Model Selection and Stepwise Regression

With 15 candidate predictors there are 32,768 possible subsets, so people search greedily: **forward selection** adds whichever variable most improves the criterion, one at a time; **backward elimination** starts with everything and drops the least useful; **stepwise** can do both. The criterion is usually **AIC** (Akaike information criterion): roughly "fit quality minus a penalty for each extra parameter" — lower is better. It's like adjusted R² but derived from likelihood and generally a stronger penalty.

Python has no built-in `stepAIC`. Two tools to know:
- Write the loop yourself: fit `smf.ols` for every remaining candidate, keep the lowest `.aic`, repeat until nothing improves it.
- `sklearn.feature_selection.SequentialFeatureSelector` does the same greedy search but ranks candidates by **cross-validated** score rather than AIC.

**Question 5.** Candidates: `Gr_Liv_Area, Year_Built, Garage_Area, Total_Bsmt_SF, Lot_Area, Full_Bath, Fireplaces, Bedroom_AbvGr, TotRms_AbvGrd, Wood_Deck_SF, Open_Porch_SF, Year_Remod_Add, Garage_Cars, First_Flr_SF, Mas_Vnr_Area`.
(a) Write a forward-selection loop that adds the lowest-AIC variable at each step until AIC stops improving. Record the order of entry. (b) Run `SequentialFeatureSelector` (forward, `scoring='neg_root_mean_squared_error'`, `cv=5`) for the same number of features and compare the chosen sets — same or different? (c) Look at the variables that entered. Some candidates carry nearly the same information (e.g., `Garage_Cars` / `Garage_Area`, `Gr_Liv_Area` / `TotRms_AbvGrd`). How does the order of entry reflect that? (d) What does the selected model's p-value table actually mean, given that the same data chose the variables and then tested them?

*Tools for this question:*
- `res.aic` (lower is better). Loop idea: fit one model per remaining candidate, keep the lowest `.aic`.
- `SequentialFeatureSelector(LinearRegression(), n_features_to_select=k, direction="forward", scoring="neg_root_mean_squared_error", cv=5).fit(X, y)`; then `.get_support()` is a boolean mask over the columns of `X`. Imports: `sklearn.feature_selection`, `sklearn.linear_model`.

_Your interpretation:_

## 6. Weighted Regression

Ordinary least squares counts every house's squared miss equally. **Weighted least squares** multiplies each squared miss by a weight first: `minimize sum of w_i * (residual_i)^2`. A house with weight 3 pulls on the line as if it appeared three times. Two common reasons to do it: observations differ in *reliability* (noisier ones should get smaller weights), or you simply care more about some observations (the book weights more recent sales more heavily, since last year's market says more about next year's price than ten-year-old sales).

Numeric picture: two houses at the same sqft, one a 2006 sale at $150k and one a 2010 sale at $180k. If the market moved up, OLS splits the difference and predicts $165k; with weights 1 for 2006 and 3 for 2010, the prediction becomes (150 + 3*180)/4 = $172.5k — pulled toward the recent price.

**Question 6.** Using `Year_Sold` (2006-2010), make a weight that increases with recency (e.g., `w = Year_Sold - 2005`). Fit `Sale_Price ~ Gr_Liv_Area + Year_Built` by OLS and by WLS (`smf.wls(..., weights=w)`). (a) Compare coefficients. Did the weights move the price-per-sqft estimate in the direction you'd expect given what you know about the market from 2006-2010? (Check by fitting OLS separately on 2006 only and 2010 only first, so you have a baseline.) (b) How do the standard errors and R² compare, and why can't you directly compare the R² values? (c) Name one situation where you'd choose WLS for *variance* reasons rather than *recency* reasons, and what you'd use as the weights.

*Tools for this question:*
- `smf.wls("y ~ x", data=df, weights=w).fit()` — `weights` is an array aligned with the rows; larger weight = that row counts more.
- `df.query("Year_Sold == 2006")` to fit on a single year.

_Your interpretation:_

---
# Exam — check your intuition

7 questions. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, a few bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Reading a simple regression
*Role lean: DS*

A teammate fits `Sale_Price ~ Garage_Area` on all 2,930 Ames homes (`Garage_Area` in square feet):

```text
                 coef  std err      t   P>|t|
Intercept    68470.35  2737.27  25.01  <0.001
Garage_Area    237.65     5.27  45.09  <0.001

n = 2930   R-squared = 0.410   Adj. R-squared = 0.410   Residual std error = 61,384
Sale_Price std dev = 79,887
```

(a) In one sentence, what does the Garage_Area coefficient say, in dollars? (b) What does the intercept mean here, and would you put it in a slide? (c) The residual standard error is about 61,384 while the raw standard deviation of Sale_Price is about 79,887. What does comparing those two numbers tell you, and how does it connect to R-squared = 0.41? (d) A stakeholder says "R-squared of 0.41 means the model is wrong 59% of the time." What is wrong with that sentence?

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — Units and rescaling
*Role lean: DS · interview*

Using the Garage_Area model above, predict what happens to each of the following if you re-fit after converting `Garage_Area` from square feet to **square meters** (1 sq m ≈ 10.76 sq ft): the slope, the intercept, R-squared, the t-statistic on the slope, and the predictions for any given house. Then: what if you instead rescaled `Sale_Price` to be in thousands of dollars? One line of reasoning per quantity is enough — if you know *why* each one moves or doesn't, you don't need to run anything.

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Extrapolation
*Role lean: DS · ML*

The same Garage_Area model predicts **$68,470** for a house with 0 sq ft of garage and **$781,411** for a house with 3,000 sq ft of garage. In the data, Garage_Area ranges from 0 to 1,488. Which of these two predictions would you trust more, and why? What specifically can go wrong when you predict outside the range of the data, even when R-squared and the residual plots inside the range look fine? How would you guard against it in a deployed model?

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — R-squared always goes up
*Role lean: DS · ML · interview*

To see what happens when you pile on useless features, a model is fit on a random **200-row** training sample using `Lot_Area`, `Year_Built`, and `k` columns of pure random noise (each generated independently of Sale_Price). Then it's scored on the other 2,730 homes it never saw:

```text
 noise_cols  R2_train  adjR2_train  RMSE_on_2730_unseen_rows
          0     0.380        0.374                     64303
          5     0.399        0.377                     65422
         20     0.461        0.394                     68275
         50     0.517        0.346                     72758
        100     0.759        0.505                     89242
        150     0.875        0.473                    133491
```

(a) Why does training R-squared climb to 0.88 with 150 columns of pure noise? (b) Adjusted R-squared is supposed to punish useless features. Does it fully protect you here? Look at what it says at k=100 versus the unseen-data RMSE at k=100 and say what you conclude about adjusted R-squared as a safeguard. (c) Interview version: "Your model has train R-squared 0.88 and performs terribly in production. Give me two explanations and tell me how you'd tell them apart." 

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — Stepwise selection is unstable
*Role lean: DS · ML · interview*

Forward selection (add the variable that most lowers AIC, one at a time) is run twice on 15 candidate numeric features — once on one random 400-row sample of the data and once on a *different*, non-overlapping 400-row sample from the same population. The order variables were chosen in:

```text
step   half A (rows 0-399)    half B (rows 400-799)
1      Total_Bsmt_SF          Gr_Liv_Area
2      Gr_Liv_Area            Year_Built
3      Year_Built             Garage_Area
4      Bedroom_AbvGr          Total_Bsmt_SF
5      Kitchen_AbvGr          Screen_Porch
6      Bsmt_Unf_SF            Year_Remod_Add
```

(a) What does this tell you about trusting the "final model" that stepwise selection hands back? (b) Someone says "the variables stepwise picked are significant (p < 0.05) so they're the true drivers of price." What is wrong with that claim — what assumption behind the p-value did the selection process break? (c) Name two approaches you would prefer for deciding which features to keep (hint: one is for prediction, one is for explanation) and say when you'd reach for each.

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_

### Exam Q6 — When does weighting help?
*Role lean: DS · Causal*

For the Garage_Area model, the spread of the residuals (how far actual prices typically miss the fitted line) differs a lot by garage size:

```text
               n  mean_garage_area  resid_sd
tercile                                     
low garage   977             247.0   37578.0
mid garage   976             473.0   46062.0
high garage  977             698.0   87874.0
```

(a) A colleague suggests weighted least squares with weights proportional to 1 / (residual variance of the group). Which group gets the larger weight, and what's the intuition in terms of "how much should one observation's miss be allowed to pull the line"? (b) Would you expect the **coefficients** to change a lot, a little, or not at all? What would you expect to change more reliably? (c) Separately: in the book, weighted regression is also used to *down-weight old observations*. Give a situation in a real analysis (not necessarily housing) where you'd choose weights for that reason rather than for unequal variance.

**Your answer (Q6):**

> _write here_

**Review (Q6):**

> _pending — filled in after you answer_

### Exam Q7 — Correlation, causation, and the coefficient
*Role lean: Causal · DS*

A product manager reads the Garage_Area model and says: "Great — every extra square foot of garage adds about $238 to a home's value, so if we convince owners to build 200 sq ft garage additions, we'll add about $47,000 each." Without running any code: (a) name at least two reasons the regression coefficient might not be the effect of *building* a garage addition, (b) say which Ames columns you'd want in the model to start addressing your reasons, and (c) say what kind of data or design (beyond a bigger regression) would let you answer the PM's actual question.

**Your answer (Q7):**

> _write here_

**Review (Q7):**

> _pending — filled in after you answer_